<a href="https://colab.research.google.com/github/jbarrasa/goingmeta/blob/main/session44/GM_S3_6_validations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install shapes-validation --index-url https://test.pypi.org/simple/

In [ ]:
pip install --index-url https://test.pypi.org/simple/ shapes-validation graphexpectations

## Optional: Create the constraints using the GraphExpectations package

In [ ]:
import graphexpectations as ge

In [ ]:
supplierExpectations = ge.Set(nodeType="Supplier")
supplierExpectations.expect_property_values_to_match_regex(property="country", regex="^[A-Za-z]+$", message="R001_INVALID_COUNTRY")
supplierExpectations.expect_number_of_outgoing_relationship_to_be_between(relationship="SUPPLIES",min=2,message="R002_LOW_PRODUCT_OFFERING")

productExpectations = ge.Set("Product")
productExpectations.expect_number_of_property_values_to_be_between(property="unitPrice", min=1, max=1,message="R003_SINGLE_PRICE")
productExpectations.expect_property_values_to_be_between(property="unitPrice", minInclusive=10, maxExclusive=500, message="R004_PRICE_LIMIT")

customerExpectations = ge.Set("Customer")
customerExpectations.expect_property_values_to_be_of_type(property="")
customerExpectations.expect_outgoing_relationship_to_connect_to_nodes_of_type(relationship="ORDERS",targetType="Order", message="R005_CUST_BAD_SCHEMA")
customerExpectations.expect_number_of_outgoing_relationship_to_be_between(relationship="ORDERS",min="1", message="R006_CUST_NO_ORDERS")

americanProducts = ge.Set(query=" (focus:Product)-[:supplied_by]->(:Supplier { country: 'USA' }) ")
americanProducts.expect_property_values_to_be_between(property="productID", minExclusive=10,message="R007_US_PROD_ID")


In [ ]:
s = ge.Suite(desc="suite of expectations for my Neo4j Northwind KG")
s.add_expectations([supplierExpectations, productExpectations]) #, productExpectations, customerExpectations, americanProducts])

In [ ]:
s.print_suite() #or .serialise()

Expectations in this Suite include 22 triples:
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

[] a sh:NodeShape ;
    sh:property [ sh:maxExclusive 500 ;
            sh:message "R004_PRICE_LIMIT" ;
            sh:minInclusive 10 ;
            sh:path <neo4j://graph.schema#unitPrice> ],
        [ sh:maxCount 1 ;
            sh:message "R003_SINGLE_PRICE" ;
            sh:minCount 1 ;
            sh:path <neo4j://graph.schema#unitPrice> ] ;
    sh:targetClass <neo4j://graph.schema#Product> .

[] a sh:NodeShape ;
    sh:property [ sh:message "R002_LOW_PRODUCT_OFFERING" ;
            sh:minCount 2 ;
            sh:path <neo4j://graph.schema#SUPPLIES> ],
        [ sh:message "R001_INVALID_COUNTRY" ;
            sh:path <neo4j://graph.schema#country> ;
            sh:pattern "^[A-Za-z]+$" ] ;
    sh:targetClass <neo4j://graph.schema#Supplier> .




In [ ]:
shapes_ttl = """

@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix n4j: <neo4j://graph.schema#> .

[] a sh:NodeShape ;
    sh:property [ sh:maxExclusive 500 ;
            sh:message "R004_PRICE_LIMIT" ;
            sh:minInclusive 5 ;
            sh:path n4j:unitPrice ],
        [ sh:maxCount 1 ;
            sh:message "R003_SINGLE_PRICE" ;
            sh:minCount 1 ;
            sh:path n4j:unitPrice ],
        [ sh:message "R002_LOW_PRODUCT_OFFERING" ;
            sh:minCount 1 ;
            sh:path [ sh:inversePath n4j:SUPPLIES ] ];
    sh:targetClass n4j:Product .

[] a sh:NodeShape ;
    sh:property
        [ sh:message "R006_CUST_NO_ORDERS" ;
            sh:minCount "1" ;
            sh:path n4j:PURCHASED ],
        [ sh:class n4j:Order ;
            sh:message "R005_CUST_BAD_SCHEMA" ;
            sh:path n4j:PURCHASED ] ;
    sh:targetClass n4j:Customer .

[] a sh:NodeShape ;
    sh:property [ sh:message "R007_US_PROD_ID" ;
            sh:minExclusive 10 ;
            sh:path n4j:productID ] ;
    sh:targetQuery " (focus:Product)<-[:SUPPLIES]-(:Supplier { country: 'USA' }) " .

[] a sh:NodeShape ;
    sh:property
        [ sh:message "R001_INVALID_COUNTRY" ;
            sh:path n4j:country ;
            sh:pattern "^[A-Za-z]+$" ] ;
    sh:targetClass n4j:Supplier .

"""


## Load the shapes, compile into cypher and run against a GraphDB

In [ ]:
from neo4j import GraphDatabase
from shapes_validation import ValidationClient, GraphConfig
from google.colab import userdata
import pandas as pd

with GraphDatabase.driver(userdata.get('NEO4J_URL'), auth=(userdata.get('NEO4J_USR'), userdata.get('NEO4J_PWD'))) as driver:
    driver.verify_connectivity()
    client = ValidationClient(driver, config=GraphConfig(handle_vocab_uris="IGNORE", graph_mode="LPG"))
    summaries = client.import_shacl(shapes_ttl)
    print("Loaded constraints:", len(summaries))

    # show the generated cypher
    # for q in client.view_cypher():
    #     print("---------------------------------")
    #     print(q["label"], q["component"], q["resultPath"])
    #     print(q["cypher"])

    # validate whole graph
    #report = client.validate()

    # validate a subgraph (node set)
    report = client.validate_nodeset([5,6])

    print("Conforms:", report.conforms)




## Print a detailed summary of violations

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.colheader_justify', 'center')
pd.set_option('display.precision', 3)

df = pd.DataFrame(report.to_rows())
display(df[['focusNode','nodeType','propertyShape','offendingValue','resultPath','severity','message']])



,focusNode,nodeType,propertyShape,offendingValue,resultPath,severity,message
0,5,Product,ValueRangeConstraintComponent,6,productID,Violation,R007_US_PROD_ID
1,6,Product,ValueRangeConstraintComponent,7,productID,Violation,R007_US_PROD_ID
